### Dataset

In [1]:
import pandas as pd

df = pd.read_csv("hotel_review.csv")
print(df.head())


                                      review  label
0                The room had a strange odor      0
1  The room was spacious and well maintained      1
2            Air conditioner was not working      0
3       Loved the modern design of the rooms      1
4     Breakfast buffet was amazing and fresh      1


### Train and Test Data split

In [2]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'],
    df['label'],
    test_size=0.2,
    stratify=df['label']
)

### Loading the BERT Tokenizer

In [3]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

d:\Bert\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Tokenize the train and test set using the Bert Tokenizer

In [4]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

### Creating the PyTorch Dataset Class

In [5]:
import torch

class ReviewDataset(torch.utils.data.Dataset):
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

### Creating the instance for the class

In [6]:
train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)

### Loading The Pretrained BERT Model

In [7]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3448.08it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

### Training parameters for the fine-tuning process

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch"
)

### Train the model 

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

d:\Bert\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.012002
2,No log,0.001501
3,No log,0.001180


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.71s/it]
d:\Bert\myenv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=150, training_loss=0.0934296162923177, metrics={'train_runtime': 122.3489, 'train_samples_per_second': 9.808, 'train_steps_per_second': 1.226, 'total_flos': 6783331896000.0, 'train_loss': 0.0934296162923177, 'epoch': 3.0})

### Evaluate the model

In [12]:
review = "The room was worst and the staff were rude"

inputs = tokenizer(review, return_tensors="pt")

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits)

print(prediction)

tensor(0)


In [15]:
model.save_pretrained("bert-hotel-sentiment")
tokenizer.save_pretrained("bert-hotel-sentiment")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


('bert-hotel-sentiment\\tokenizer_config.json',
 'bert-hotel-sentiment\\tokenizer.json')

### Checking with the saved model

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-hotel-sentiment")

model = BertForSequenceClassification.from_pretrained(
    "bert-hotel-sentiment"
)

model.eval()

In [18]:
review = "The room was extremely dirty and the staff were very childish"

inputs = tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

In [20]:
import torch

with torch.no_grad():
    outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=1).item()
if pred == 1:
    print("Positive Review")
else:
    print("Negative Review")

Negative Review
